# Assess the fine-tuned Cellpose model

This notebook compares the default Cellpose ('cyto') model with your fine-tuned model(s) using aggregate segmentation metrics. The output is limited to summary tables; no per-image results are displayed.


# Imports

In [1]:
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from cellpose import core, io, metrics, models
from IPython.display import display

io.logger_setup()
if not core.use_gpu():
    raise ImportError("No GPU access; use a VIBE instance with GPU support")

2026-11-04 11:56:48,023 [INFO] WRITING LOG OUTPUT TO /storage/homefs/rl18y689/.cellpose/run.log
2026-11-04 11:56:48,024 [INFO] 
cellpose version: 	3.1.1.3 
platform:       	linux 
python version: 	3.11.18 
torch version:  	2.15.0+cu132
2026-11-04 11:56:48,246 [INFO] ** TORCH CUDA version installed and working. **


# Configuration and load the validation split

We will load the validation (unseen) dataset to asses the models 
performance. By default we are usign the split that contains 500 images.
 If you want to asses the metrics with a different split size, adjust it
 accordingly.

In [2]:
dataset_name = "A172_splits_n500_seed42"
iou_thresholds = [0.5, 0.75, 0.9]

data_dir = Path("../data_cp")
models_dir = Path("../models")
model_backbone = "cyto"

In [3]:
validation_dir = data_dir / dataset_name / "validation"
if not validation_dir.is_dir():
    raise FileNotFoundError(
        f"Could not find validation split for {dataset_name}: {validation_dir.resolve()}"
    )

validation_image_paths = sorted(validation_dir.glob("*.tif"))
if not validation_image_paths:
    raise FileNotFoundError(f"No .tif images found in {validation_dir.resolve()}")

validation_mask_paths = [validation_dir / f"{path.stem}_seg.npy" for path in validation_image_paths]
missing_masks = [path for path in validation_mask_paths if not path.is_file()]
if missing_masks:
    raise FileNotFoundError(f"Missing ground-truth mask: {missing_masks[0].resolve()}")

validation_images = [io.imread(str(path)) for path in validation_image_paths]
validation_labels = [np.load(path, allow_pickle=True).item()["masks"] for path in validation_mask_paths]

for image_path, image, label in zip(validation_image_paths, validation_images, validation_labels):
    if image.shape[:2] != label.shape[:2]:
        raise ValueError(
            f"Image/mask shape mismatch for {image_path}: {image.shape} vs {label.shape}"
        )

print(f"Loaded {len(validation_images)} validation images and ground-truth masks from {validation_dir.resolve()}")

Loaded 50 validation images and ground-truth masks from /storage/research/dsl_vibe_rs/trainings/2026_deep_learning_with_VIBE/cellpose/data_cp/A172_splits_n500_seed42/validation


## Load the models and generate predictions

We will try to load all models we have trained (in case we have more than one) and gather hyperparameters information so that we can compare them later on  on the summary table

## Find the models

In [4]:
# Cellpose saves fine-tuned models as single files (usually without extension)
model_paths = sorted(
    p for p in models_dir.iterdir()
    if p.is_file() and not p.name.startswith(".")
)
if not model_paths:
    raise FileNotFoundError(f"No model files found in {models_dir.resolve()}")

print("***************************")
print(f"Found {len(model_paths)} fine-tuned model(s) in {models_dir}:")
for p in model_paths:
    print(f"  - {p.name}")
print("***************************")

***************************
Found 7 fine-tuned model(s) in ../models:
  - A172_splits_n100_seed42_cyto-finetuned_n_epochs10_b_size4_w_decay0.1_l_rate1e-05
  - A172_splits_n100_seed42_cyto-finetuned_n_epochs50_b_size32_w_decay0.1_l_rate1e-05
  - A172_splits_n100_seed42_cyto-finetuned_n_epochs51_b_size1_w_decay0.1_l_rate1e-05
  - A172_splits_n100_seed42_cyto-finetuned_n_epochs51_b_size4_w_decay0.1_l_rate1e-05
  - A172_splits_n200_seed42_cyto-finetuned_n_epochs101_b_size64_w_decay0.1_l_rate1e-05
  - A172_splits_n200_seed42_cyto-finetuned_n_epochs50_b_size64_w_decay0.1_l_rate1e-05
  - A172_splits_n500_seed42_cyto-finetuned_n_epochs101_b_size64_w_decay0.1_l_rate1e-05
***************************


## Hyperparameter parser

In [5]:
# column name -> (regex with one capture group, type to cast to)
HYPERPARAM_PATTERNS = {
    "Training set": (r"^(.*)_[A-Za-z0-9]+-finetuned", str),
    "Backbone": (r"_([A-Za-z0-9]+)-finetuned", str),
    "Epochs": (r"n_epochs(\d+)", int),
    "Batch size": (r"b_size(\d+)", int),
    "Weight decay": (r"w_decay([0-9.eE+-]+)", float),
    "Learning rate": (r"l_rate([0-9.eE+-]+)", float),
}


def parse_hyperparameters(model_name):
    """Extract training hyperparameters encoded in a model name; missing ones become None."""
    params = {}
    for column, (pattern, cast) in HYPERPARAM_PATTERNS.items():
        match = re.search(pattern, model_name)
        params[column] = cast(match.group(1)) if match else None
    return params


# Quick sanity check
display(pd.DataFrame([{"Model": p.name, **parse_hyperparameters(p.name)} for p in model_paths]))

,Model,Training set,Backbone,Epochs,Batch size,Weight decay,Learning rate
0,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,10,4,0.1,0.00001
1,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,50,32,0.1,0.00001
2,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,51,1,0.1,0.00001
3,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,51,4,0.1,0.00001
4,A172_splits_n200_seed42_cyto-finetuned_n_epoch...,A172_splits_n200_seed42,cyto,101,64,0.1,0.00001
5,A172_splits_n200_seed42_cyto-finetuned_n_epoch...,A172_splits_n200_seed42,cyto,50,64,0.1,0.00001
6,A172_splits_n500_seed42_cyto-finetuned_n_epoch...,A172_splits_n500_seed42,cyto,101,64,0.1,0.00001


## Predictions

In [6]:
def predict(model, images, **eval_kwargs):
    return [model.eval(image, **eval_kwargs)[0] for image in images]


def free_gpu(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


DEFAULT_LABEL = f"Default backbone '{model_backbone}'"
predictions = {}

# Baseline
default_model = models.CellposeModel(gpu=True, pretrained_model=model_backbone)
predictions[DEFAULT_LABEL] = predict(default_model, validation_images, channels=[0, 0])
free_gpu(default_model)

# Fine-tuned models
for path in model_paths:
    try:
        model = models.CellposeModel(gpu=True, pretrained_model=str(path))
    except Exception as e:
        print(f"Skipping {path.name}: could not load ({e})")
        continue
    print(f"Evaluating {path.name}")
    predictions[path.name] = predict(model, validation_images)
    free_gpu(model)

2026-11-04 11:56:50,682 [INFO] >> cyto << model set to be used
2026-11-04 11:56:50,683 [INFO] ** TORCH CUDA version installed and working. **
2026-11-04 11:56:50,683 [INFO] >>>> using GPU (CUDA)
2026-11-04 11:56:50,736 [INFO] >>>> loading model /storage/homefs/rl18y689/.vibe/.cellpose/models/cytotorch_0
2026-11-04 11:56:51,039 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)


/opt/.pixi/envs/cellpose-gui-default/lib/python3.11/site-packages/cellpose/dynamics.py:760: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:916.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),


2026-11-04 11:57:00,766 [INFO] ** TORCH CUDA version installed and working. **
2026-11-04 11:57:00,766 [INFO] >>>> using GPU (CUDA)
2026-11-04 11:57:00,810 [INFO] >>>> loading model ../models/A172_splits_n100_seed42_cyto-finetuned_n_epochs10_b_size4_w_decay0.1_l_rate1e-05
2026-11-04 11:57:00,896 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)
2026-11-04 11:57:00,897 [INFO] >>>> model diam_labels =  34.419 (mean diameter of training ROIs)
Evaluating A172_splits_n100_seed42_cyto-finetuned_n_epochs10_b_size4_w_decay0.1_l_rate1e-05
2026-11-04 11:57:14,844 [INFO] ** TORCH CUDA version installed and working. **
2026-11-04 11:57:14,845 [INFO] >>>> using GPU (CUDA)
2026-11-04 11:57:14,875 [INFO] >>>> loading model ../models/A172_splits_n100_seed42_cyto-finetuned_n_epochs50_b_size32_w_decay0.1_l_rate1e-05
2026-11-04 11:57:14,978 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)
2026-11-04 11:57:14,979 [INFO] >>>> model diam_lab

## Summarize segmentation performance

### What the metrics mean

- **Intersection over Union (IoU):** overlap between a predicted object mask and its ground-truth mask, calculated as intersection area divided by union area. A predicted/true object pair counts as a match only if its IoU reaches the selected threshold.
- **Mean AP @ 0.50 / 0.75 / 0.90:** in Cellpose, AP is the object-level score `TP / (TP + FP + FN)` at the specified IoU threshold. The notebook computes it for each image and averages those scores across images. Higher is better. This is Cellpose's thresholded mask-matching AP, not the area under a confidence-ranked precision-recall curve.
- **Precision @ 0.50:** `TP / (TP + FP)`, using TP and FP summed over all validation images. Of the predicted objects, this is the fraction matched to ground truth. Higher means fewer unmatched/extra predictions.
- **Recall @ 0.50:** `TP / (TP + FN)`, using counts summed over the validation images. Of the ground-truth objects, this is the fraction detected. Higher means fewer missed objects.
- **F1 @ 0.50:** harmonic mean of the reported precision and recall, `2 * precision * recall / (precision + recall)`. It balances missed and extra objects; higher is better.
- **Mean absolute object-count error:** average across images of `abs(number of predicted objects - number of ground-truth objects)`. Lower is better, but it does not measure whether object boundaries or identities match.

These metrics summarize different properties and should be interpreted together. All matching metrics use IoU thresholds, so a visually close mask may still count as a miss if its overlap falls below the threshold.

### References

1. Cellpose `average_precision` implementation and IoU-based object matching: [MouseLand/cellpose, `cellpose/metrics.py`](https://github.com/MouseLand/cellpose/blob/main/cellpose/metrics.py).
2. Stringer C, Wang T, Michaelos M, Pachitariu M. *Cellpose: a generalist algorithm for cellular segmentation.* Nature Methods 18, 100-106 (2021). [https://doi.org/10.1038/s41592-020-01018-x](https://doi.org/10.1038/s41592-020-01018-x).

The table contains only model-level summary values; no per-image metric table is generated.

In [7]:
def summarize_predictions(model_label, predictions):
    if len(predictions) != len(validation_labels):
        raise ValueError(
            f"{model_label} produced {len(predictions)} masks for {len(validation_labels)} validation labels"
        )

    ap, true_positive, false_positive, false_negative = metrics.average_precision(
        validation_labels,
        predictions,
        threshold=iou_thresholds,
    )
    tp = true_positive[:, 0].sum()
    fp = false_positive[:, 0].sum()
    fn = false_negative[:, 0].sum()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    true_object_counts = [len(np.unique(mask)) - 1 for mask in validation_labels]
    predicted_object_counts = [len(np.unique(mask)) - 1 for mask in predictions]

    return {
        "Model": model_label,
        "Images": len(validation_labels),
        "Mean AP @ 0.50": ap[:, 0].mean(),
        "Mean AP @ 0.75": ap[:, 1].mean(),
        "Mean AP @ 0.90": ap[:, 2].mean(),
        "Precision @ 0.50": precision,
        "Recall @ 0.50": recall,
        "F1 @ 0.50": f1,
        "Mean abs. object-count error": np.mean(
            np.abs(np.asarray(predicted_object_counts) - np.asarray(true_object_counts))
        ),
    }


rows = []
for label, preds in predictions.items():
    hyperparams = parse_hyperparameters(label)
    if label == DEFAULT_LABEL:
        hyperparams["Backbone"] = model_backbone
    rows.append({**summarize_predictions(label, preds), **hyperparams})

summary = pd.DataFrame(rows)

hyperparam_columns = list(HYPERPARAM_PATTERNS)
metric_columns = [c for c in summary.columns if c not in ["Model", "Images", *hyperparam_columns]]
summary = summary[["Model", *hyperparam_columns, "Images", *metric_columns]]

# Round metrics only; rounding would turn small learning rates such as 1e-05 into 0
summary[metric_columns] = summary[metric_columns].astype(float).round(3)
summary = summary.sort_values("Mean AP @ 0.50", ascending=False, ignore_index=True)
display(summary)

,Model,Training set,Backbone,Epochs,Batch size,Weight decay,Learning rate,Images,Mean AP @ 0.50,Mean AP @ 0.75,Mean AP @ 0.90,Precision @ 0.50,Recall @ 0.50,F1 @ 0.50,Mean abs. object-count error
0,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,10.0,4.0,0.1,0.00001,50,0.592,0.324,0.040,0.778,0.591,0.671,43.70
1,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,51.0,4.0,0.1,0.00001,50,0.589,0.322,0.040,0.770,0.592,0.669,42.12
2,A172_splits_n500_seed42_cyto-finetuned_n_epoch...,A172_splits_n500_seed42,cyto,101.0,64.0,0.1,0.00001,50,0.539,0.276,0.029,0.732,0.551,0.629,44.86
3,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,51.0,1.0,0.1,0.00001,50,0.494,0.195,0.013,0.721,0.494,0.586,56.60
4,A172_splits_n200_seed42_cyto-finetuned_n_epoch...,A172_splits_n200_seed42,cyto,101.0,64.0,0.1,0.00001,50,0.489,0.230,0.017,0.686,0.516,0.589,45.28
5,A172_splits_n200_seed42_cyto-finetuned_n_epoch...,A172_splits_n200_seed42,cyto,50.0,64.0,0.1,0.00001,50,0.479,0.220,0.014,0.667,0.512,0.579,42.72
6,A172_splits_n100_seed42_cyto-finetuned_n_epoch...,A172_splits_n100_seed42,cyto,50.0,32.0,0.1,0.00001,50,0.477,0.211,0.011,0.646,0.510,0.570,38.66
7,Default backbone 'cyto',NaN,cyto,NaN,NaN,NaN,NaN,50,0.282,0.132,0.014,0.636,0.245,0.354,110.80
